In [ ]:
import os
os.chdir("..")
%pwd

In [ ]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class ModelTrainerConfig:
    root_dir: Path
    train_data_path: Path
    test_data_path: Path
    model_name: str
    target_column: str


@dataclass(frozen=True)
class ModelEvaluationConfig:
    root_dir: Path
    test_data_path: Path
    model_path: Path
    metric_file_name: Path
    target_column: str

In [ ]:
from src.datascience.constants import CONFIG_FILE_PATH, PARAMS_FILE_PATH, SCHEMA_FILE_PATH
from src.datascience.utils.common import read_yaml, create_directories, save_json

In [ ]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath=CONFIG_FILE_PATH,
        params_filepath=PARAMS_FILE_PATH,
        schema_filepath=SCHEMA_FILE_PATH,
    ):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)
        create_directories([self.config.artifacts_root])

    def get_model_trainer_config(self) -> ModelTrainerConfig:
        config = self.config.model_trainer
        create_directories([config.root_dir])
        return ModelTrainerConfig(
            root_dir=Path(config.root_dir),
            train_data_path=Path(config.train_data_path),
            test_data_path=Path(config.test_data_path),
            model_name=config.model_name,
            target_column=self.schema.TARGET_COLUMN.name,
        )

    def get_model_evaluation_config(self) -> ModelEvaluationConfig:
        config = self.config.model_evaluation
        create_directories([config.root_dir])
        return ModelEvaluationConfig(
            root_dir=Path(config.root_dir),
            test_data_path=Path(config.test_data_path),
            model_path=Path(config.model_path),
            metric_file_name=Path(config.metric_file_name),
            target_column=self.schema.TARGET_COLUMN.name,
        )

In [ ]:
import joblib
import pandas as pd
import matplotlib.pyplot as plt
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
)
from src.datascience import logger

In [ ]:
class ModelTrainer:
    def __init__(self, config: ModelTrainerConfig):
        self.config = config

    def train(self):
        train_df = pd.read_csv(self.config.train_data_path)
        test_df  = pd.read_csv(self.config.test_data_path)

        X_train = train_df.drop(columns=[self.config.target_column])
        y_train = train_df[self.config.target_column]
        X_test  = test_df.drop(columns=[self.config.target_column])
        y_test  = test_df[self.config.target_column]

        # Compute scale_pos_weight from data to handle class imbalance
        neg = (y_train == 0).sum()
        pos = (y_train == 1).sum()
        scale_pos_weight = neg / pos
        logger.info(f"neg: {neg} | pos: {pos} | scale_pos_weight: {scale_pos_weight:.2f}")

        params = read_yaml(PARAMS_FILE_PATH)
        param_grid = {
            'n_estimators':     list(params.XGBoost.n_estimators),
            'max_depth':        list(params.XGBoost.max_depth),
            'learning_rate':    list(params.XGBoost.learning_rate),
            'subsample':        list(params.XGBoost.subsample),
            'colsample_bytree': list(params.XGBoost.colsample_bytree),
        }

        xgb = XGBClassifier(
            scale_pos_weight=scale_pos_weight,
            eval_metric='logloss',
            random_state=42,
            verbosity=0,
        )

        gs = GridSearchCV(
            estimator=xgb,
            param_grid=param_grid,
            cv=params.GridSearchCV.cv,
            scoring=params.GridSearchCV.scoring,
            n_jobs=params.GridSearchCV.n_jobs,
            verbose=1,
        )

        logger.info("Starting GridSearchCV")
        gs.fit(X_train, y_train)

        logger.info(f"Best params:     {gs.best_params_}")
        logger.info(f"Best CV ROC AUC: {gs.best_score_:.4f}")

        model_path = Path(self.config.root_dir) / self.config.model_name
        joblib.dump(gs.best_estimator_, model_path)
        logger.info(f"Model saved -> {model_path}")

        return gs.best_estimator_

In [ ]:
class ModelEvaluation:
    def __init__(self, config: ModelEvaluationConfig):
        self.config = config

    def evaluate(self):
        test_df = pd.read_csv(self.config.test_data_path)
        X_test  = test_df.drop(columns=[self.config.target_column])
        y_test  = test_df[self.config.target_column]

        model  = joblib.load(self.config.model_path)
        y_pred = model.predict(X_test)
        y_prob = model.predict_proba(X_test)[:, 1]

        roc_auc  = roc_auc_score(y_test, y_prob)
        accuracy = accuracy_score(y_test, y_pred)
        report   = classification_report(y_test, y_pred, output_dict=True)
        cm       = confusion_matrix(y_test, y_pred)

        metrics = {
            'roc_auc':          round(roc_auc,  4),
            'accuracy':         round(accuracy, 4),
            'precision':        round(report['weighted avg']['precision'], 4),
            'recall':           round(report['weighted avg']['recall'],    4),
            'f1':               round(report['weighted avg']['f1-score'],  4),
            'confusion_matrix': cm.tolist(),
        }

        save_json(self.config.metric_file_name, metrics)
        logger.info(f"ROC AUC: {metrics['roc_auc']}")
        logger.info(f"F1:      {metrics['f1']}")

        # ROC curve plot
        fpr, tpr, _ = roc_curve(y_test, y_prob)
        plt.figure(figsize=(8, 6))
        plt.plot(fpr, tpr, color='darkorange', lw=2,
                 label=f'ROC curve (AUC = {roc_auc:.4f})')
        plt.plot([0, 1], [0, 1], color='navy', lw=1, linestyle='--',
                 label='Random classifier')
        plt.fill_between(fpr, tpr, alpha=0.1, color='darkorange')
        plt.xlim([0.0, 1.0])
        plt.ylim([0.0, 1.05])
        plt.xlabel('False Positive Rate', fontsize=13)
        plt.ylabel('True Positive Rate', fontsize=13)
        plt.title('ROC Curve — Bid Bot Detection', fontsize=14)
        plt.legend(loc='lower right', fontsize=12)
        plt.tight_layout()

        roc_path = Path(self.config.root_dir) / 'roc_curve.png'
        plt.savefig(roc_path, dpi=150)
        plt.show()
        logger.info(f"ROC curve saved -> {roc_path}")

        return metrics

In [ ]:
# --- Train ---
try:
    config = ConfigurationManager()
    model_trainer_config = config.get_model_trainer_config()
    model_trainer = ModelTrainer(config=model_trainer_config)
    model_trainer.train()
except Exception as e:
    from src.datascience import BidBotException
    import sys
    raise BidBotException(e, sys)

In [ ]:
# --- Evaluate ---
try:
    config = ConfigurationManager()
    model_evaluation_config = config.get_model_evaluation_config()
    model_evaluation = ModelEvaluation(config=model_evaluation_config)
    metrics = model_evaluation.evaluate()
    print(metrics)
except Exception as e:
    from src.datascience import BidBotException
    import sys
    raise BidBotException(e, sys)